# **Classification of *Solanum betaceum* Cav Maturity**


# 1. Dataset Preparation

## 1.1 Image Format Normalization

In [ ]:
!unzip "/project-2-Berenjena.zip" -d /project_Berenjena2

In [ ]:
import os
from PIL import Image
from tqdm import tqdm

def convert_and_resize_images(source_folder, destination_folder, new_width=640, new_height=640):
    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)

    files = [file for file in os.listdir(source_folder) if os.path.isfile(os.path.join(source_folder, file)) and
                file.lower().endswith(('.png', '.jpeg', '.bmp', '.gif', '.tiff', 'jpg','JPG'))]

    counter = 1

    with tqdm(total=len(files), desc="Processing images", unit="image") as pbar:
        for file in files:
            file_path = os.path.join(source_folder, file)

            try:
                image = Image.open(file_path)

                image = image.resize((640, 640), Image.Resampling.LANCZOS)

                output_name = f"epp_{counter}.jpg"
                output_path = os.path.join(destination_folder, output_name)

                image.convert('RGB').save(output_path, 'jpg')

                counter += 1
            except Exception as e:
                pass

            pbar.update(1)

source_folder = "project_Berenjena2"
destination_folder = "Data_Berenjena2"

convert_and_resize_images(source_folder, destination_folder)

# 2. Class Labeling Performed in Label-Studio

Images were processed in Label-Studio, with each class being assigned its corresponding label. An example of the file format is provided below:

```
0 0.217969 0.441667 0.091146 0.129630
0 0.332552 0.242593 0.043229 0.100000
0 0.291406 0.125463 0.031771 0.060185
0 0.238802 0.138426 0.031771 0.060185
0 0.297396 0.049074 0.023958 0.038889
0 0.320052 0.019907 0.017188 0.030556
0 0.465104 0.316204 0.055208 0.125000
0 0.495052 0.482407 0.067187 0.177778
0 0.141406 0.965278 0.116146 0.069444
```

It can be observed that the coordinates of the bounding box for each object are normalized to the image size:

![](https://github.com/ultralytics/docs/releases/download/0/two-persons-tie.avif)

# 3. Dataset Structure

According to Ultralytics documentation, the image set should be structured as follows:

![](https://github.com/ultralytics/docs/releases/download/0/two-persons-tie-2.avif)

## 3.1. Creation of the data.yaml Class List

In [ ]:

import yaml

data = {
    'path': 'project_Berenjena2',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: "green fruit",
        1: "ripe fruit",
       }
}

with open('project_Berenjena2/data.yaml', 'w') as file:
    yaml.dump(data, file,
              default_flow_style=False,
              sort_keys=False)

## 3.2. Dataset Division into Training, Testing, and Validation Sets (Cross-validation K=5)

In [ ]:
import os
import shutil
from sklearn.model_selection import KFold

# ==========================================================
# CONFIGURATION
# ==========================================================

base_dir = "project_Berenjena2"

images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

output_dir = os.path.join(base_dir, "KFold")

k = 5
random_state = 42

# Class names
class_names = ["green fruit","ripe fruit"
]

# ==========================================================
# RETRIEVE IMAGES
# ==========================================================

image_extensions = (".jpg", ".jpeg", ".png")

image_files = sorted([
    f for f in os.listdir(images_dir)
    if f.lower().endswith(image_extensions)
])

print(f"Total images: {len(image_files)}")

# ==========================================================
# K-FOLD
# ==========================================================

kf = KFold(
    n_splits=k,
    shuffle=True,
    random_state=random_state
)

for fold, (train_idx, val_idx) in enumerate(kf.split(image_files), start=1):

    print(f"\n======================")
    print(f"Fold {fold}")
    print("======================")

    fold_dir = os.path.join(output_dir, f"fold_{fold}")

    train_images = os.path.join(fold_dir, "images", "train")
    val_images   = os.path.join(fold_dir, "images", "val")

    train_labels = os.path.join(fold_dir, "labels", "train")
    val_labels   = os.path.join(fold_dir, "labels", "val")

    for folder in [
        train_images,
        val_images,
        train_labels,
        val_labels
    ]:
        os.makedirs(folder, exist_ok=True)

    for idx in train_idx:

        image_name = image_files[idx]

        label_name = os.path.splitext(image_name)[0] + ".txt"

        shutil.copy2(
            os.path.join(images_dir, image_name),
            os.path.join(train_images, image_name)
        )

        if os.path.exists(os.path.join(labels_dir, label_name)):
            shutil.copy2(
                os.path.join(labels_dir, label_name),
                os.path.join(train_labels, label_name)
            )

    # -----------------------------
    # Copy VALIDATION
    # -----------------------------
    for idx in val_idx:

        image_name = image_files[idx]

        label_name = os.path.splitext(image_name)[0] + ".txt"

        shutil.copy2(
            os.path.join(images_dir, image_name),
            os.path.join(val_images, image_name)
        )

        if os.path.exists(os.path.join(labels_dir, label_name)):
            shutil.copy2(
                os.path.join(labels_dir, label_name),
                os.path.join(val_labels, label_name)
            )

    # ======================================================
    # CREATE DATA.YAML
    # ======================================================

    yaml_path = os.path.join(fold_dir, "data.yaml")

    with open(yaml_path, "w") as f:

        f.write(f"path: {fold_dir}\n")
        f.write("train: images/train\n")
        f.write("val: images/val\n\n")

        f.write(f"nc: {len(class_names)}\n")

        f.write("names:\n")

        for i, name in enumerate(class_names):
            f.write(f"  {i}: {name}\n")

    print(f"Train: {len(train_idx)}")
    print(f"Val  : {len(val_idx)}")

print("\n======================================")
print("K-Fold generated successfully.")
print("======================================")

## 3.3. Data augmentation applied only to the training folds

In [ ]:
import random
import csv
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import numpy as np
import yaml

# ──────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────
YAML_PATH = r"project_Berenjena3\fold_1\data.yaml"

STRATEGY = "both"

MAX_AUGMENT_COPIES = 4

MAX_RATIO = 1

SEED = 42

# ──────────────────────────────────────────────
# UTILITIES
# ──────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)


def load_yaml(yaml_path: str) -> dict:
    with open(yaml_path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def count_instances(label_dir: Path) -> tuple[Counter, dict]:

    instance_counter: Counter = Counter()
    class_to_images: dict = defaultdict(list)

    for label_file in sorted(label_dir.rglob("*.txt")):
        classes_in_file = set()
        with open(label_file, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls = int(parts[0])
                instance_counter[cls] += 1
                classes_in_file.add(cls)
        for cls in classes_in_file:
            class_to_images[cls].append(label_file)

    return instance_counter, dict(class_to_images)


def print_distribution(counter: Counter, names: dict, title: str = "Distribution"):
    print(f"\n{'─'*50}")
    print(f"  {title}")
    print(f"{'─'*50}")
    total = sum(counter.values())
    for cls_id, cls_name in names.items():
        n = counter.get(cls_id, 0)
        pct = 100 * n / total if total else 0
        bar = "█" * int(pct / 2)
        print(f"  [{cls_id}] {cls_name:<20} {n:>6} instances  ({pct:5.1f}%)  {bar}")
    print(f"{'─'*50}")
    print(f"  Total instances: {total}")


# ──────────────────────────────────────────────
# IMAGE + LABEL AUGMENTATION
# ──────────────────────────────────────────────
def augment_image(img: np.ndarray) -> list[np.ndarray]:
    augmented = []

    # Flip horizontal
    augmented.append(cv2.flip(img, 1))

    # Flip vertical
    augmented.append(cv2.flip(img, 0))

    # Random brightness
    factor = random.uniform(0.6, 1.4)
    bright = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)
    augmented.append(bright)

    # Mild Gaussian noise
    noise = np.random.normal(0, 8, img.shape).astype(np.float32)
    noisy = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    augmented.append(noisy)

    return augmented


def augment_labels(label_path: Path, aug_index: int) -> list[str]:

    lines = label_path.read_text(encoding="utf-8").splitlines()
    new_lines = []
    for line in lines:
        parts = line.split()
        if not parts:
            continue
        cls, x, y, w, h = parts[0], float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        if aug_index == 0:   # horizontal flip
            x = 1.0 - x
        elif aug_index == 1:  # vertical flip
            y = 1.0 - y
        new_lines.append(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    return new_lines


def resolve_image_path(label_path: Path, img_dir: Path, label_dir: Path) -> Path | None:

    rel = label_path.relative_to(label_dir)
    for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
        candidate = img_dir / rel.with_suffix(ext)
        if candidate.exists():
            return candidate
    return None


# ──────────────────────────────────────────────
# OVER-SAMPLING
# ──────────────────────────────────────────────
def oversample_class(
    cls_id: int,
    label_files: list[Path],
    img_dir: Path,
    label_dir: Path,
    target_copies: int,
):
    if not label_files:
        return 0

    added = 0
    aug_pool = list(range(min(MAX_AUGMENT_COPIES, 4)))

    files_cycle = label_files.copy()
    random.shuffle(files_cycle)

    idx = 0
    while added < target_copies:
        label_path = files_cycle[idx % len(files_cycle)]
        aug_i = aug_pool[added % len(aug_pool)]

        img_path = resolve_image_path(label_path, img_dir, label_dir)
        if img_path is None:
            idx += 1
            added += 1
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            idx += 1
            added += 1
            continue

        aug_imgs = augment_image(img)
        aug_img = aug_imgs[aug_i]

        rel_folder = label_path.parent.relative_to(label_dir)
        target_img_dir = img_dir / rel_folder
        target_img_dir.mkdir(parents=True, exist_ok=True)

        target_label_dir = label_dir / rel_folder
        target_label_dir.mkdir(parents=True, exist_ok=True)

        new_stem = f"{label_path.stem}_aug{cls_id}_{added:04d}"
        new_img_path = target_img_dir / (new_stem + img_path.suffix)
        new_label_path = target_label_dir / (new_stem + ".txt")

        cv2.imwrite(str(new_img_path), aug_img)
        new_lines = augment_labels(label_path, aug_i)
        new_label_path.write_text("\n".join(new_lines), encoding="utf-8")

        added += 1
        idx += 1

    return added


# ──────────────────────────────────────────────
# UNDER-SAMPLING
# ──────────────────────────────────────────────
def undersample_class(
    cls_id: int,
    label_files: list[Path],
    img_dir: Path,
    label_dir: Path,
    target_count: int,
    names: dict,
):
    """Remove images (and their labels) belonging to a majority class."""
    current = len(label_files)
    to_remove = current - target_count
    if to_remove <= 0:
        return 0

    aug_files = [f for f in label_files if "_aug" in f.stem]
    orig_files = [f for f in label_files if "_aug" not in f.stem]
    candidates = aug_files + orig_files
    random.shuffle(candidates)

    removed = 0
    for label_path in candidates:
        if removed >= to_remove:
            break

        img_path = resolve_image_path(label_path, img_dir, label_dir)
        if img_path and img_path.exists():
            img_path.unlink()

        if label_path.exists():
            label_path.unlink()

        removed += 1

    print(f"    Under-sampling [{cls_id}] {names[cls_id]}: -{removed} images removed")
    return removed


# ──────────────────────────────────────────────
# SAVE REPORT CSV
# ──────────────────────────────────────────────
def save_report(
    split: str,
    before: Counter,
    after: Counter,
    names: dict,
    output_csv: Path,
):
    rows = []
    for cls_id, cls_name in names.items():
        rows.append({
            "split": split,
            "class_id": cls_id,
            "class_name": cls_name,
            "before": before.get(cls_id, 0),
            "after": after.get(cls_id, 0),
            "delta": after.get(cls_id, 0) - before.get(cls_id, 0),
        })
    write_header = not output_csv.exists()
    with open(output_csv, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["split", "class_id", "class_name", "before", "after", "delta"])
        if write_header:
            writer.writeheader()
        writer.writerows(rows)
    print(f"\n  ✔ Report saved to: {output_csv}")


# ──────────────────────────────────────────────
# MAIN PIPELINE
# ──────────────────────────────────────────────
def balance_dataset(dataset_path: Path, names: dict, strategy: str):
    """
    Balance ONLY the training dataset.
    Uses:
      - images/train/ + labels/train/
    """
    # <-- ONLY CHANGE: Point specifically to the 'train' folder -->
    img_dir = dataset_path / "images" / "train"
    label_dir = dataset_path / "labels" / "train"
    # ------------------------------------------------------------------

    print(f"\n{'═'*50}")
    print(f"  Dataset: TRAIN")
    print(f"  Images : {img_dir}")
    print(f"  Labels : {label_dir}")
    print(f"{'═'*50}")

    if not img_dir.exists() or not label_dir.exists():
        print("  ⚠ Train directories not found. Check dataset structure.")
        return

    before_counter, class_to_imgs = count_instances(label_dir)
    print_distribution(before_counter, names, title="BEFORE balancing — TRAIN")

    if not before_counter:
        print("  No instances found.")
        return

    counts = {cls: before_counter.get(cls, 0) for cls in names}
    mean_count = int(np.mean(list(counts.values())))
    max_count = max(counts.values())
    min_count = min(counts.values())

    print(f"\n  Mean instances : {mean_count}")
    print(f"  Max instances  : {max_count}")
    print(f"  Min instances  : {min_count}")
    print(f"  Max/min ratio  : {max_count / (min_count or 1):.2f}x")

    # Over-sampling
    if strategy in ("oversample", "both"):
        print("\n  [OVER-SAMPLING] Minority classes →")
        target_os = int(mean_count * 1.0)
        for cls_id in names:
            current = counts.get(cls_id, 0)
            if current < target_os:
                label_files = class_to_imgs.get(cls_id, [])
                if not label_files:
                    print(f"    [{cls_id}] {names[cls_id]}: no base images. Skipping.")
                    continue
                deficit_imgs = len(label_files)
                copies_needed = max(1, (target_os - current) // max(deficit_imgs, 1))
                copies_needed = min(copies_needed, MAX_AUGMENT_COPIES)
                added = oversample_class(
                    cls_id,
                    label_files,
                    img_dir,
                    label_dir,
                    target_copies=copies_needed * deficit_imgs,
                )
                print(f"    [{cls_id}] {names[cls_id]:<22} +{added} images generated")
            else:
                print(f"    [{cls_id}] {names[cls_id]:<22} OK (no over-sampling required)")

    # Recount after over-sampling
    after_os_counter, class_to_imgs = count_instances(label_dir)

    # Under-sampling
    if strategy in ("undersample", "both"):
        print("\n  [UNDER-SAMPLING] Majority classes →")
        all_counts = [after_os_counter.get(c, 0) for c in names]
        new_mean = int(np.mean(all_counts))
        target_us = int(new_mean * MAX_RATIO)
        for cls_id in names:
            current = after_os_counter.get(cls_id, 0)
            if current > target_us:
                label_files = class_to_imgs.get(cls_id, [])
                if label_files:
                    target_count = max(1, int(len(label_files) * (target_us / current)))
                    undersample_class(
                        cls_id,
                        label_files,
                        img_dir,
                        label_dir,
                        target_count=target_count,
                        names=names,
                    )
                else:
                    print(f"    [{cls_id}] {names[cls_id]:<22} OK (no files for under-sampling)")
            else:
                print(f"    [{cls_id}] {names[cls_id]:<22} OK (no under-sampling required)")

    # Final analysis
    after_counter, _ = count_instances(label_dir)
    print_distribution(after_counter, names, title="AFTER balancing — TRAIN")

    final_counts = [after_counter.get(c, 0) for c in names]
    if min(final_counts) > 0:
        final_ratio = max(final_counts) / min(final_counts)
        print(f"\n  Final max/min ratio: {final_ratio:.2f}x")

    # Save report
    report_path = dataset_path / "balance_report.csv"
    save_report("train", before_counter, after_counter, names, report_path) # <-- Changed "all" to "train"


def main():
    cfg = load_yaml(YAML_PATH)
    dataset_path = Path(cfg["path"])
    names: dict = cfg["names"]

    print("\n" + "═" * 50)
    print("  YOLO Dataset Class Balancer (Train Only)")
    print(f"  Dataset : {dataset_path}")
    print(f"  Classes  : {names}")
    print(f"  Strategy: {STRATEGY}")
    print("═" * 50)

    balance_dataset(dataset_path, names, STRATEGY)

    print("\n✔ Balancing process completed.\n")


if __name__ == "__main__":
    main()

# 4. YOLO11 (Detection) Training with the Dataset

To access the pre-trained model, we must first install the "ultralytics" library. Additionally, the following models are available:

![](https://drive.google.com/uc?export=view&id=1PYaeIKj57C4_nys6XPAJrZ2tfdwiYEZQ)

In all cases, the maximum image dimension to be used is 640 pixels; therefore, the images in our dataset will need to be adjusted to this size.

In [ ]:
!pip install ultralytics

## 4.1. GFLOPs

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.info(imgsz=640)  # reports parameters and GFLOPs

YOLO11n summary: 181 layers, 2,624,080 parameters, 0 gradients, 6.6 GFLOPs


(181, 2624080, 0, 6.614336)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.info(imgsz=640)  # reports parameters and GFLOPs

YOLO11s summary: 181 layers, 9,458,752 parameters, 0 gradients, 21.7 GFLOPs


(181, 9458752, 0, 21.718374400000002)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11m.pt")
model.info(imgsz=640)  # reports parameters and GFLOPs

YOLO11m summary: 231 layers, 20,114,688 parameters, 0 gradients, 68.5 GFLOPs


(231, 20114688, 0, 68.52838399999999)

Subsequently, the model is trained:

1. Import the pre-trained model.
2. Use the "train" method to perform the training.

## Training and Hyperparameter Tuning

In [ ]:
from ultralytics import YOLO
import os

# ==========================================================
# CONFIGURATION
# ==========================================================

base_dir = "/project_Berenjena2/KFold"

k = 5

# ==========================================================
# K-FOLD TRAINING
# ==========================================================

for fold in range(1, k + 1):

    print("=" * 60)
    print(f"TRAINING FOLD {fold}/{k}")
    print("=" * 60)

    # Reinitialize the model for each fold
    model = YOLO("yolo11m.pt")

    yaml_file = os.path.join(
        base_dir,
        f"fold_{fold}",
        "data.yaml"
    )

    results = model.train(

        # Dataset
        data=yaml_file,

        # Training
        epochs=300,# Change to 100, 200, and 300
        imgsz=640,
        batch=32,

        # Optimizer
        optimizer="AdamW",

        # Learning rate
        lr0=0.001,
        lrf=0.01,
        cos_lr=True,

        # Regularization
        weight_decay=0.0005,

        # Warmup
        warmup_epochs=3,
        warmup_momentum=0.8,

        # Data augmentation
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=10,
        translate=0.1,
        scale=0.5,
        shear=2,
        perspective=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,

        # Performance
        cache="ram",
        workers=8,
        amp=True,

        # Saving
        save=True,
        plots=True,
        verbose=True,

        # Results folder
        project="runs/KFold",

        # Experiment name
        name=f"YOLO11s300_Fold_{fold}",

        # Prevents errors if the folder already exists
        exist_ok=True
    )

print("\nK-Fold training finished.")

# 5. Generating Predictions with the Model

Finally, we can use the test set (available in the folder `../datasets/vehicle_dataset/images/test`) to generate predictions on new data using the fine-tuned model.

Let's start by loading the fine-tuned model:

In [ ]:
model = YOLO('/content/runs/detect/train2/weights/best.pt')

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Create subplots with less space between them
fig, axes = plt.subplots(3, 3, figsize=(12, 12), constrained_layout=True)

# Define margin size (in pixels)
margin_size = 10

# Iterate over subplots and display images
for i, ax in enumerate(axes.flat):
    if i < len(preds):
        image = preds[i].plot(line_width=14, font_size=70)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert from BGR to RGB

        # Add a black border around the image
        image_with_border = cv2.copyMakeBorder(
            image, margin_size, margin_size, margin_size, margin_size,
            borderType=cv2.BORDER_CONSTANT, value=[128, 128, 128] # Black color
        )

        ax.imshow(image_with_border)
        ax.axis("off")
    else:
        ax.axis("off")

# Adjust spacing automatically
plt.tight_layout()
plt.subplots_adjust(wspace=0.01, hspace=0.01)
plt.savefig('Predictions.jpg', dpi=900)
plt.show()

# 6. Figures

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.ticker import MaxNLocator
from pathlib import Path
from PIL import Image
import random

# CONFIGURATION

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 19,
    "axes.titlesize": 22,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "axes.spines.top": False,
    "axes.spines.right": False
})

base_path = Path(r"project_Berenjena2")

subset = "" # subset

img_dir = base_path / "images" / subset
label_dir = base_path / "labels" / subset

if not img_dir.exists():
    img_dir = base_path / "images"
    label_dir = base_path / "labels"

class_names = {0: "Green fruit",
               1: "Ripe fruit"}


# DATASET READING

fruits_per_image = [] # fruits_per_image
class_counts = {0:0, 1:0} # class_counts
images_processed = 0 # images_processed

example_image = None # example_image

for image_file_path in sorted(img_dir.iterdir()): # image_file

    if image_file_path.suffix.lower() not in [".jpg",".jpeg",".png"]:
        continue

    if example_image is None:
        example_image = image_file_path

    label_file_path = label_dir / f"{image_file_path.stem}.txt" # label_file

    green_count = 0 # green
    ripe_count = 0 # ripe

    if label_file_path.exists():

        with open(label_file_path) as f:

            for line in f:

                data_parts = line.split() # data

                if len(data_parts) >= 5:

                    class_id = int(data_parts[0]) # class

                    if class_id == 0:
                        green_count += 1
                        class_counts[0]+=1

                    elif class_id == 1:
                        ripe_count +=1
                        class_counts[1]+=1

    fruits_per_image.append(green_count + ripe_count)
    images_processed +=1

# STATISTICS

mean_fruits = np.mean(fruits_per_image) # mean
median_fruits = np.median(fruits_per_image) # median

num_samples = 100 # num_samples
if images_processed > num_samples:
    image_indices = sorted([i+1 for i in random.sample(range(images_processed), num_samples)]) # image_indices
else:
    image_indices = list(range(1, images_processed + 1))

fruits_per_image = [fruits_per_image[i-1] for i in image_indices]

# FIGURE

fig, ax = plt.subplots(1, 2,
                       figsize=(14, 7),
                       dpi=900,
                       gridspec_kw={'width_ratios': [2, 1]})

# SEQUENTIAL PLOT

ax[0].plot(image_indices, fruits_per_image,
           color="#4F81BD", marker='o', markersize=5,
           linestyle='-', linewidth=1.2, label="Fruits count")

ax[0].fill_between(image_indices, fruits_per_image, alpha=0.15, color="#4F81BD")

ax[0].axhline(mean_fruits, color="red", linewidth=2, linestyle="--", label=f"Mean = {mean_fruits:.2f}")
ax[0].axhline(median_fruits, color="green", linewidth=2, linestyle="-.", label=f"Median = {median_fruits:.0f}")

ax[0].set_xlabel("Image index")
ax[0].set_ylabel("Number of fruits per image")
ax[0].set_title("(a) Distribution of fruits per image", pad=28)

ax[0].grid(True, linestyle="--", alpha=0.50)
ax[0].xaxis.set_major_locator(MaxNLocator(integer=True))
ax[0].yaxis.set_major_locator(MaxNLocator(integer=True))
ax[0].legend(frameon=False, loc='upper right')

# INSERT FRUIT IMAGE

if example_image is not None:

    img = Image.open(example_image)
    img.thumbnail((230,230))

    imagebox = OffsetImage(img, zoom=0.35)

    ab = AnnotationBbox(
        imagebox,
        (0.15, 0.80),
        xycoords='axes fraction',
        frameon=True,
        bboxprops=dict(edgecolor='black', boxstyle="round,pad=0.3")
    )

    ax[0].add_artist(ab)

# CLASS BALANCE

classes_list = list(class_names.values()) # classes
totals_list = list(class_counts.values()) # totals

colors_list = ["forestgreen", "tomato"] # colors

bars = ax[1].bar(
    classes_list,
    totals_list,
    color=colors_list,
    edgecolor="black",
    width=0.55
)

ax[1].set_ylabel("Number of labeled objects")
ax[1].set_title("(b) Class distribution", pad=28)
ax[1].grid(axis="y", linestyle="--", alpha=0.35)
ax[1].yaxis.set_major_locator(MaxNLocator(integer=True))

total_count = sum(totals_list)

for bar, class_total in zip(bars, totals_list): # class_total

    percentage = 100 * class_total / total_count # percentage

    ax[1].text(
        bar.get_x() + bar.get_width()/2,
        class_total + max(totals_list)*0.02,
        f"{class_total}\n({percentage:.1f}%)",
        ha="center",
        fontsize=13,
        fontweight="bold"
    )

# SAVE AND DISPLAY

plt.tight_layout()

plt.savefig("Figure 2.png", dpi=900, bbox_inches="tight")
plt.savefig("Figure 2.pdf", bbox_inches="tight")
plt.show()

# SUMMARY

print("="*50)
print("DATASET SUMMARY")
print("="*50)
print(f"Images                 : {images_processed}")
print(f"Average fruits/image   : {mean_fruits:.2f}")
print(f"Median fruits/image    : {median_fruits:.2f}")
print(f"Green fruits           : {class_counts[0]}")
print(f"Ripe fruits            : {class_counts[1]}")

In [ ]:
import warnings
import os
import pickle
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')

# CONFIGURATION

PROJECT_DIR = r"Yolo11-300"
FOLD_TO_PLOT = 2
EPOCHS = 300

MODELS = {
    "YOLO11n": f"runs_detect_runs_KFold_YOLO11n{EPOCHS}_Fold_{FOLD_TO_PLOT}",
    "YOLO11s": f"runs_detect_runs_KFold_YOLO11s{EPOCHS}_Fold_{FOLD_TO_PLOT}",
    "YOLO11m": f"runs_detect_runs_KFold_YOLO11m{EPOCHS}_Fold_{FOLD_TO_PLOT}",
}

COLORS = {
    "YOLO11n": "#1f77b4", # Blue
    "YOLO11s": "#2ca02c", # Green
    "YOLO11m": "#d62728", # Red
}

DASHES = {'YOLO11n': '-', 'YOLO11s': '--', 'YOLO11m': '-.'}
MARKS  = {'YOLO11n': 'o', 'YOLO11s': 's',  'YOLO11m': '^'}
DPI    = 900
OUTPUT_DIR = Path('eval_convergence_plots')
CACHE_FILE = Path('data_cache.pkl')

# STYLE

def set_style():
    style = 'seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'whitegrid'
    plt.style.use(style)
    plt.rcParams.update({
        'font.family':       'serif',
        'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
        'font.size':         14,
        'axes.titlesize':    16,
        'axes.titleweight':  'bold',
        'axes.labelsize':    14,
        'xtick.labelsize':   12,
        'ytick.labelsize':   12,
        'legend.fontsize':   11,
        'lines.linewidth':   1.5,
        'figure.facecolor':  'white',
        'axes.facecolor':    'white',
    })

def save_fig(fig, name: str):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for ext in ['jpg', 'pdf']:
        filepath = OUTPUT_DIR / f'{name}.{ext}'
        fig.savefig(filepath, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.close(fig)

# DATA EXTRACTION

def load_csv_data():
    print(f"Extracting data ({EPOCHS} epochs, Fold {FOLD_TO_PLOT})...")
    data = {}
    for model_name, folder_name in MODELS.items():
        csv_path = os.path.join(PROJECT_DIR, folder_name, "results.csv")
        if not os.path.exists(csv_path):
            print(f"⚠️ Not found: {csv_path}")
            continue

        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()

        if "epoch" not in df.columns:
            df.insert(0, "epoch", range(len(df)))
            df = df.dropna(subset=['epoch'])
        data[model_name] = df

    return data

# PLOTS (TRAINING AND VALIDATION)

def plot_reviewer_curves(data: dict):
    if len(data) < 1: return
    names = list(data.keys())
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
    fig.suptitle(f'Convergence Analysis ({EPOCHS} Epochs - Fold {FOLD_TO_PLOT})', fontsize=18, fontweight='bold')

    # Plot 1: Train vs Val Loss
    for n in names:
        df = data[n]
        if 'train/box_loss' not in df or 'val/box_loss' not in df: continue
        ax1.plot(df['epoch'], df['train/box_loss'], color=COLORS[n], linestyle='-', linewidth=2, label=f'{n} Train')
        ax1.plot(df['epoch'], df['val/box_loss'], color=COLORS[n], linestyle='--', linewidth=2, label=f'{n} Val')

    ax1.set_title('Training and Validation Loss Convergence')
    ax1.set_xlabel('Epochs'); ax1.set_ylabel('Box Loss')
    ax1.legend(loc='upper right'); ax1.set_xlim(0, EPOCHS)

    # Plot 2: mAP50-95
    for n in names:
        df = data[n]
        if 'metrics/mAP50-95(B)' not in df: continue
        ax2.plot(df['epoch'], df['metrics/mAP50-95(B)'], color=COLORS[n], linestyle='-', linewidth=2.5, label=n)
        ax2.scatter(df['epoch'].iloc[-1], df['metrics/mAP50-95(B)'].iloc[-1],
                    s=65, marker=MARKS[n], color=COLORS[n], zorder=5, edgecolor='k', linewidth=0.5)

    ax2.set_title('Validation mAP@0.5:0.95 Convergence')
    ax2.set_xlabel('Epochs'); ax2.set_ylabel('mAP50-95')
    ax2.legend(loc='lower right'); ax2.set_xlim(0, EPOCHS)

    plt.tight_layout()
    save_fig(fig, f'fig_convergence_fold{FOLD_TO_PLOT}')
    print(f"  [saved] fig_convergence_fold{FOLD_TO_PLOT}")

#----------------------------------------------------------
def plot_loss_detail(data: dict):
    if len(data) < 1: return
    names = list(data.keys())
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Detailed Training and Validation Losses', fontsize=18, fontweight='bold')

    loss_pairs = [
        ('train/box_loss', 'val/box_loss', 'Box Loss'),
        ('train/cls_loss', 'val/cls_loss', 'Classification Loss'),
        ('train/dfl_loss', 'val/dfl_loss', 'DFL Loss'),
    ]

    for col, (tk, vk, lbl) in enumerate(loss_pairs):
        for row, (key, split) in enumerate([(tk, 'Train'), (vk, 'Val')]):
            ax = axes[row, col]
            ax.set_title(f'{split} — {lbl}')
            for n in names:
                df = data[n]
                if key not in df: continue
                ax.plot(df['epoch'], df[key], DASHES[n], color=COLORS[n], label=n, linewidth=1.5)
            ax.set_xlim(0, EPOCHS)
            ax.set_xlabel('Epochs'); ax.set_ylabel('Loss')

    handles = [Line2D([0],[0], color=COLORS[n], linestyle=DASHES[n], label=n) for n in names]
    fig.legend(handles=handles, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05), frameon=False, fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    save_fig(fig, 'fig_detailed_loss')
    print("  [saved] fig_detailed_loss")


# COMPARATIVE PLOT: VALIDATION VS TEST
def plot_train_val_test_comparison(data: dict):

    test_data = {}
    if CACHE_FILE.exists():
        with open(CACHE_FILE, 'rb') as f:
            test_data = pickle.load(f)

    if not test_data:
        print("  [skipped] Test comparison")
        return

    names = [n for n in data.keys() if n in test_data]
    if not names: return

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Train, Validation (Epoch {EPOCHS}) vs. Test Set', fontsize=16, fontweight='bold')

    x = np.arange(len(names))
    width = 0.25

    # --- PANEL 1: mAP@50:95 (Val vs Test) ---

    val_maps, test_maps = [], []
    for n in names:
        df = data[n]
        val_map = df['metrics/mAP50-95(B)'].dropna().iloc[-1] if 'metrics/mAP50-95(B)' in df else 0
        val_maps.append(val_map)
        test_maps.append(test_data[n]['mAP95'])

    bars1 = axes[0].bar(x - width/2, val_maps, width, label='Validation (Last Epoch)', color='#1f77b4', alpha=0.85, edgecolor='black')
    bars2 = axes[0].bar(x + width/2, test_maps, width, label='Test Set', color='#d62728', alpha=0.85, edgecolor='black', hatch='//')

    axes[0].set_title('mAP@0.5:0.95 (Val vs Test)')
    axes[0].set_xticks(x); axes[0].set_xticklabels(names)
    axes[0].set_ylabel('Score'); axes[0].set_ylim(0, max(max(val_maps), max(test_maps)) * 1.2)
    axes[0].legend(frameon=False, loc='upper left')

    for bar in bars1 + bars2:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold')

    # --- PANEL 2: Box Loss (Train vs Val) to check Overfitting ---
    tr_losses, val_losses = [], []
    for n in names:
        df = data[n]
        tr_l = df['train/box_loss'].dropna().iloc[-1] if 'train/box_loss' in df else 0
        val_l = df['val/box_loss'].dropna().iloc[-1] if 'val/box_loss' in df else 0
        tr_losses.append(tr_l)
        val_losses.append(val_l)

    bars3 = axes[1].bar(x - width/2, tr_losses, width, label='Train (Last Epoch)', color='#2ca02c', alpha=0.85, edgecolor='black')
    bars4 = axes[1].bar(x + width/2, val_losses, width, label='Validation (Last Epoch)', color='#1f77b4', alpha=0.85, edgecolor='black', hatch='//')

    axes[1].set_title('Final Box Loss (Overfitting Check)')
    axes[1].set_xticks(x); axes[1].set_xticklabels(names)
    axes[1].set_ylabel('Loss')
    axes[1].legend(frameon=False, loc='upper right')

    for bar in bars3 + bars4:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    save_fig(fig, 'fig_train_val_test_comparison')
    print("  [saved] fig_train_val_test_comparison")


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    data = load_csv_data()

    if len(data) > 0:
        set_style()
        print(f"\nGenerating figures (Fold {FOLD_TO_PLOT}, {EPOCHS} Epochs)..\n")

        # 1. Main curves (Train vs Val)
        plot_reviewer_curves(data)

        # 2. Detailed loss (Train vs Val)
        plot_loss_detail(data)

        # 3. Direct comparison (Train vs Val vs Test)
        plot_train_val_test_comparison(data)

        print(f"\n✅ All figures saved to: {OUTPUT_DIR.resolve()}")
    else:
        print("\n❌ Error: Could not load model data from CSVs.")

In [ ]:
import warnings
import pickle
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')

# STYLE AND OUTPUT CONFIGURATION

CACHE_FILE = Path('data_cache.pkl')
OUTPUT_DIR = Path('eval_convergence_plots')

COLORS = {'YOLO11n': '#1f77b4', 'YOLO11s': '#ff7f0e', 'YOLO11m': '#d62728'}
DASHES = {'YOLO11n': '-',       'YOLO11s': '--',       'YOLO11m': '-.'}
MARKS  = {'YOLO11n': 'o',       'YOLO11s': 's',        'YOLO11m': '^'}
DPI    = 900

def set_style():
    rcParams.update({
        'font.family':       'serif',
        'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
        'font.size':         14,
        'axes.titlesize':    14,
        'axes.titleweight':  'bold',
        'axes.labelsize':    13,
        'xtick.labelsize':   12,
        'ytick.labelsize':   12,
        'legend.fontsize':   12,
        'lines.linewidth':   1.3,
        'axes.spines.top':   False,
        'axes.spines.right': False,
        'axes.grid':         True,
        'grid.linewidth':    0.3,
        'grid.alpha':        0.5,
        'figure.facecolor':  'white',
        'axes.facecolor':    'white',
    })

def save_fig(fig, out: Path, name: str):
    """Saves the figure in JPG and PDF."""
    for ext in ['jpg', 'pdf']:
        filepath = out / f'{name}.{ext}'
        fig.savefig(filepath, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.close(fig)

# PLOTTING FUNCTIONS

def plot_reviewer_curves(data: dict, out: Path):
    names = list(data.keys())
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

    fig.suptitle('Convergence Analysis', fontsize=16, fontweight='bold')

    ax1.set_title('Training vs Validation Loss')
    for n in names:
        ep = data[n]['epoch']
        if len(ep) == 0: continue
        ax1.plot(ep, data[n]['tr_box'], DASHES[n], color=COLORS[n], label=f'{n} Train', alpha=0.9)
        ax1.plot(ep, data[n]['val_box'], DASHES[n], color=COLORS[n], label=f'{n} Val', alpha=0.9, linestyle='--')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Box Loss')
    ax1.legend(frameon=False, loc='upper right', fontsize=11)

    ax2.set_title('Validation mAP@0.5:0.95 Convergence')
    for n in names:
        ep, y = data[n]['epoch'], data[n]['map95_ep']
        if len(ep) == 0: continue
        ax2.plot(ep, y, DASHES[n], color=COLORS[n], label=n, linewidth=2)
        ax2.scatter(ep[-1], y[-1], s=65, marker=MARKS[n], color=COLORS[n], zorder=5, edgecolor='k', linewidth=0.5)

    ax2.set_xlabel('Epoch', fontsize=16); ax2.set_ylabel('mAP50-95', fontsize=16)
    ax2.legend(frameon=False, loc='lower right')

    save_fig(fig, out, 'fig_convergence_reviewer')
    print("  [saved] fig_convergence_reviewer")

#----------------------------------------------------------
def plot_loss_detail(data: dict, out: Path):
    names = list(data.keys())
    fig, axes = plt.subplots(2, 3, figsize=(11, 6), constrained_layout=True)
    fig.suptitle('Training and Validation Losses', fontsize=18, fontweight='bold')

    loss_pairs = [
        ('tr_box', 'val_box', 'Box Loss'),
        ('tr_cls', 'val_cls', 'Classification Loss'),
        ('tr_dfl', 'val_dfl', 'DFL Loss'),
    ]

    for col, (tk, vk, lbl) in enumerate(loss_pairs):
        for row, (key, split) in enumerate([(tk, 'Train'), (vk, 'Val')]):
            ax = axes[row, col]; ax.set_title(f'{split} — {lbl}')
            for n in names:
                ep, y = data[n]['epoch'], data[n][key]
                if len(ep) == 0 or len(y) == 0: continue
                ax.plot(ep, y, DASHES[n], color=COLORS[n], label=n, alpha=0.9)
            ax.set_xlabel('Epoch', fontsize=16); ax.set_ylabel('Loss', fontsize=16)
            ax.tick_params(axis='both', labelsize=16)

    handles = [Line2D([0],[0], color=COLORS[n], linestyle=DASHES[n], label=n) for n in names]
    fig.legend(handles=handles, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.06), frameon=False, fontsize=14)

    save_fig(fig, out, 'fig_detailed_loss')
    print("  [saved] fig_detailed_loss")

#----------------------------------------------------------
def plot_perclass_test(data: dict, out: Path):
    names = list(data.keys())
    if len(names) < 1 or not data[names[0]].get('per_class'): return

    all_classes = list(data[names[0]]['per_class'].keys())
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    fig.suptitle('Test Set Metrics by Class', fontsize=18, fontweight='bold')

    for ax, metric, label in [(axes[0],'ap50','mAP@50'), (axes[1],'ap','mAP@50:95')]:
        x, w = np.arange(len(all_classes)), 0.26
        for i, n in enumerate(names):
            if n not in data or not data[n].get('per_class'): continue
            vals = [data[n]['per_class'][c][metric] for c in all_classes]
            bars = ax.bar(x + (i-1)*w, vals, w, label=n, color=COLORS.get(n,'grey'), alpha=0.88, edgecolor='white')
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=12)
        ax.set_title(label); ax.set_xticks(x)
        ax.set_xticklabels(all_classes, rotation=20, ha='right', fontsize=12)
        ax.set_ylabel('Score'); ax.set_ylim(0, 1.15); ax.legend(frameon=False, loc='upper right', fontsize=12, bbox_to_anchor=(0.55, -0.1))

    save_fig(fig, out, 'Test Set Metrics by Class')
    print("  [saved] fig_test_perclass")

#----------------------------------------------------------
def plot_accuracy_efficiency_bubble(data: dict, out: Path):
    names = list(data.keys())
    fig, ax = plt.subplots(figsize=(7, 5))
    fig.suptitle('Performance vs. Model Complexity (Test Set)', fontsize=14, fontweight='bold')

    for n in names:
        d = data[n]
        x = d['params_M']
        y = d['mAP95']
        s = d['mAP50'] * 600

        ax.scatter(x, y, s=s, color=COLORS[n], alpha=0.7, edgecolors='black', linewidth=0.8, zorder=3)
        ax.annotate(f"{n}\n({x:.2f}M)", (x, y),
                    xytext=(0, 20), textcoords='offset points',
                    ha='center', fontweight='bold', color=COLORS[n], fontsize=11)

    ax.set_xlabel('Model Parameters (M)', fontsize=14)
    ax.set_ylabel('Test mAP@0.5:0.95', fontsize=14)
    ax.text(0.95, 0.05, 'Bubble size \u221d mAP@0.50',
            transform=ax.transAxes, fontsize=11, ha='right', color='grey', style='italic')

    save_fig(fig, out, 'Detection Performance versus Model Complexity (Test Set)')
    print("  [saved] fig_accuracy_vs_complexity")

#----------------------------------------------------------
def plot_global_metrics_summary(data: dict, out: Path):
    names = list(data.keys())
    metrics_to_plot = {'Precision': 'P', 'Recall': 'R', 'mAP@50': 'mAP50', 'mAP@50:95': 'mAP95'}
    y_pos = np.arange(len(metrics_to_plot))
    height = 0.25

    fig, ax = plt.subplots(figsize=(8, 4))
    fig.suptitle('Quantitative Comparison of YOLO11 Models on the Test Set', fontsize=14, fontweight='bold')

    for i, n in enumerate(names):
        values = [data[n][k] for k in metrics_to_plot.values()]
        bars = ax.barh(y_pos + (i-1)*height, values, height, label=n, color=COLORS[n], alpha=0.85)
        for bar, val in zip(bars, values):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
                    va='center', ha='left', fontsize=11, color='black')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(list(metrics_to_plot.keys()))
    ax.set_xlim(0, 1.1)
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(0.98, 0.98))
    ax.invert_yaxis()

    save_fig(fig, out, 'Quantitative Comparison of YOLO11 Models on the Test Set')
    print("  [saved] fig_global_metrics_bar")



# MAIN

def main():
    if not CACHE_FILE.exists():
        print(f"[ERROR] File {CACHE_FILE} does not exist.")
        print("Please run '1_extract_data.py' first.")
        return

    print("Loading data from cache...")
    with open(CACHE_FILE, 'rb') as f:
        data = pickle.load(f)

    out = OUTPUT_DIR
    out.mkdir(parents=True, exist_ok=True)
    set_style()

    plot_reviewer_curves(data, out)
    plot_loss_detail(data, out)
    plot_perclass_test(data, out)
    plot_accuracy_efficiency_bubble(data, out)
    plot_global_metrics_summary(data, out)

    print(f"\n\u2705 All figures saved to: {out.resolve()}")

if __name__ == '__main__':
    main()